> **Production note (2026-06-21):** The script pipeline in `scripts/` is the source of truth for final outputs. This notebook is retained for exploration and narrative context; run the README pipeline for reproducible delivery artifacts.


# Exploratory Data Analysis - Biodiesel Consumption
## Repsol Capstone Project - Week 1

Analysis of biodiesel consumption patterns in Spain (2023-2025)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

NOTEBOOK_DIR = Path().resolve()
REPO_ROOT    = NOTEBOOK_DIR.parent
DATA_INPUTS  = REPO_ROOT / 'data' / 'inputs'
FIGS         = REPO_ROOT / 'reports' / 'figures'
FIGS.mkdir(parents=True, exist_ok=True)

# Load cleaned biodiesel consumption: all CCAAs + ESPAÑA national total, 2023-2025
df = pd.read_csv(DATA_INPUTS / 'consumo_biodiesel_ccaa.csv')
df['Fecha'] = pd.to_datetime(df['Fecha'])
df = df.sort_values('Fecha').reset_index(drop=True)

print(f"Dataset: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Columns: {df.columns.tolist()}")
print(f"Date range: {df['Fecha'].min().strftime('%Y-%m')} \u2192 {df['Fecha'].max().strftime('%Y-%m')}")
print(f"CCAAs ({df['CCAA'].nunique()}): {sorted(df['CCAA'].unique())}")


## 1. Setup — Data Loading

Data loaded from `data/inputs/consumo_biodiesel_ccaa.csv` (720 rows, 3 columns: Fecha, CCAA, Consumo_Tm).

In [ ]:
print("=" * 60)
print("DATA EXPLORATION - BIODIESEL CONSUMPTION")
print("=" * 60)

print(f"\nDIMENSIONS:")
print(f"   Rows: {len(df)}")
print(f"   Period: {df['Fecha'].min().strftime('%Y-%m')} to {df['Fecha'].max().strftime('%Y-%m')}")
print(f"   CCAA/Nacional: {df['CCAA'].nunique()}")

print(f"\nCONSUMPTION STATISTICS (Tm):")
print(df['Consumo_Tm'].describe())

print(f"\nMISSING VALUES:")
print(df.isnull().sum())

print(f"\nFIRST 10 ROWS:")
print(df.head(10))


## 2. National Consumption

In [ ]:
# 1. NATIONAL CONSUMPTION OVER TIME
df_national = df[df['CCAA'] == 'ESPAÑA'].copy()

fig, axes = plt.subplots(2, 1, figsize=(15, 10))

# Chart 1: National time series
axes[0].plot(df_national['Fecha'], df_national['Consumo_Tm'], 
             linewidth=2, marker='o', color='#FF6B35', markersize=4)
axes[0].set_title('Biodiesel Consumption in Spain (2023-2025)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Fecha')
axes[0].set_ylabel('Consumption (Tm)')
axes[0].grid(True, alpha=0.3)
axes[0].fill_between(df_national['Fecha'], df_national['Consumo_Tm'], alpha=0.2, color='#FF6B35')

# Chart 2: Monthly distribution
axes[1].bar(df_national['Fecha'], df_national['Consumo_Tm'], 
            color='#004E89', width=20, alpha=0.7)
axes[1].set_title('Monthly Consumption (Bars)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Fecha')
axes[1].set_ylabel('Consumption (Tm)')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(FIGS / '01_consumo_nacional.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Chart saved: 01_consumo_nacional.png")

## 3. Regional Analysis — Top 5 CCAA

In [ ]:
# 2. CONSUMPTION BY AUTONOMOUS COMMUNITY (TOP 5)
df_regional = df[df['CCAA'] != 'ESPAÑA'].copy()

# Calculate total consumption by CCAA
ccaa_consumption = df_regional.groupby('CCAA')['Consumo_Tm'].sum().sort_values(ascending=False)

print("🏆 TOP 10 CCAA by total consumption (2023-2025):")
print(ccaa_consumption.head(10))

# Visualize TOP 5
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Chart 1: Top 5 CCAA
top5_ccaa = ccaa_consumption.head(5)
axes[0].barh(top5_ccaa.index, top5_ccaa.values, color='#1f77b4')
axes[0].set_title('Top 5 CCAA - Total Biodiesel Consumption (2023-2025)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Total Consumption (Tm)')
axes[0].grid(True, alpha=0.3, axis='x')

# Chart 2: Top 5 time series
for ccaa in top5_ccaa.index:
    df_ccaa = df[df['CCAA'] == ccaa].sort_values('Fecha')
    axes[1].plot(df_ccaa['Fecha'], df_ccaa['Consumo_Tm'], 
                 marker='o', label=ccaa, linewidth=2, markersize=3)

axes[1].set_title('Time Evolution - Top 5 CCAA', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Fecha')
axes[1].set_ylabel('Consumption (Tm)')
axes[1].legend(loc='best')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGS / '02_consumo_regional.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Chart saved: 02_consumo_regional.png")

## 4. Seasonality y Trend

In [ ]:
# 3. SEASONALITY AND TREND ANALYSIS
df_national = df[df['CCAA'] == 'ESPAÑA'].copy()

# Extract month and year
df_national['Mes'] = df_national['Fecha'].dt.month
df_national['Año'] = df_national['Fecha'].dt.year
df_national['Mes_Nombre'] = df_national['Fecha'].dt.strftime('%B')

# Average consumption by month (seasonality)
consumo_por_mes = df_national.groupby('Mes')['Consumo_Tm'].mean()
mes_nombres = ['January', 'February', 'March', 'April', 'May', 'June',
               'July', 'August', 'September', 'October', 'November', 'December']

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Chart 1: Seasonality (average by month)
axes[0, 0].bar(range(1, 13), consumo_por_mes.values, color='#2ecc71', alpha=0.7)
axes[0, 0].set_title('Seasonality - Average Consumption by Month', fontsize=13, fontweight='bold')
axes[0, 0].set_xlabel('Month')
axes[0, 0].set_ylabel('Average Consumption (Tm)')
axes[0, 0].set_xticks(range(1, 13))
axes[0, 0].set_xticklabels([m[:3] for m in mes_nombres], rotation=45)
axes[0, 0].grid(True, alpha=0.3, axis='y')

# Chart 2: Consumption by year
consumo_por_año = df_national.groupby('Año')['Consumo_Tm'].sum()
axes[0, 1].bar(consumo_por_año.index, consumo_por_año.values, color='#e74c3c', alpha=0.7, width=0.6)
axes[0, 1].set_title('Total Consumption by Year', fontsize=13, fontweight='bold')
axes[0, 1].set_xlabel('Year')
axes[0, 1].set_ylabel('Total Consumption (Tm)')
axes[0, 1].set_xticks(consumo_por_año.index)
axes[0, 1].grid(True, alpha=0.3, axis='y')

# Chart 3: Box plot by month (variability)
df_national_sorted = df_national.sort_values('Mes')
df_national_sorted['Mes_Nombre_Short'] = df_national_sorted['Mes'].map({i: m[:3] for i, m in enumerate(mes_nombres, 1)})

box_data = [df_national[df_national['Mes'] == i]['Consumo_Tm'].values for i in range(1, 13)]
bp = axes[1, 0].boxplot(box_data, labels=[m[:3] for m in mes_nombres], patch_artist=True)
for patch in bp['boxes']:
    patch.set_facecolor('#3498db')
    patch.set_alpha(0.7)
axes[1, 0].set_title('Variabilidad Monthly (Box Plot)', fontsize=13, fontweight='bold')
axes[1, 0].set_xlabel('Month')
axes[1, 0].set_ylabel('Consumption (Tm)')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Chart 4: Overall distribution
axes[1, 1].hist(df_national['Consumo_Tm'], bins=20, color='#9b59b6', alpha=0.7, edgecolor='black')
axes[1, 1].set_title('Distribution of Total Consumption', fontsize=13, fontweight='bold')
axes[1, 1].set_xlabel('Consumption (Tm)')
axes[1, 1].set_ylabel('Frecuencia')
axes[1, 1].axvline(df_national['Consumo_Tm'].mean(), color='red', linestyle='--', linewidth=2, label=f'Media: {df_national["Consumo_Tm"].mean():.0f} Tm')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(FIGS / '03_estacionalidad_tendencia.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Chart saved: 03_estacionalidad_tendencia.png")

# Statistics by month
print("\n📊 AVERAGE CONSUMPTION BY MONTH:")
for mes, valor in consumo_por_mes.items():
    print(f"   {mes_nombres[mes-1]}: {valor:.0f} Tm")

print("\n📊 TOTAL CONSUMPTION BY YEAR:")
for año, valor in consumo_por_año.items():
    print(f"   {año}: {valor:.0f} Tm")

## 5. Patrones Temporales

In [ ]:
# 4. CORRELATION AND FINAL STATISTICAL SUMMARY
df_national = df[df['CCAA'] == 'ESPAÑA'].copy()

# Create time features
df_national['Mes'] = df_national['Fecha'].dt.month
df_national['Trimestre'] = df_national['Fecha'].dt.quarter
df_national['Año'] = df_national['Fecha'].dt.year
df_national['Día_Año'] = df_national['Fecha'].dt.dayofyear

# Correlation matrix
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Chart 1: Scatter - Mes vs Consumption
axes[0].scatter(df_national['Mes'], df_national['Consumo_Tm'], 
                s=100, alpha=0.6, c=df_national['Año'], cmap='viridis')
axes[0].set_title('Consumption by Month (colored by Year)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Consumption (Tm)')
axes[0].set_xticks(range(1, 13))
axes[0].grid(True, alpha=0.3)
cbar = plt.colorbar(axes[0].collections[0], ax=axes[0])
cbar.set_label('Year')

# Chart 2: Consumption by quarter
consumo_trimestre = df_national.groupby('Trimestre')['Consumo_Tm'].mean()
colores_trim = ['#e74c3c', '#f39c12', '#2ecc71', '#3498db']
axes[1].bar(consumo_trimestre.index, consumo_trimestre.values, 
            color=colores_trim, alpha=0.7, width=0.6)
axes[1].set_title('Average Consumption by Quarter', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Quarter')
axes[1].set_ylabel('Average Consumption (Tm)')
axes[1].set_xticks([1, 2, 3, 4])
axes[1].set_xticklabels(['Q1', 'Q2', 'Q3', 'Q4'])
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(FIGS / '04_correlaciones.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Chart saved: 04_correlaciones.png")

# RESUMEN FINAL
print("\n" + "="*60)
print("FINAL STATISTICAL SUMMARY - EDA COMPLETED")
print("="*60)

print(f"\n📈 GENERAL DATA:")
print(f"   Period: {df_national['Fecha'].min().strftime('%Y-%m-%d')} to {df_national['Fecha'].max().strftime('%Y-%m-%d')}")
print(f"   Total months: {len(df_national)}")
print(f"   Consumption total Spain (3 años): {df_national['Consumo_Tm'].sum():,.0f} Tm")
print(f"   Consumption average monthly: {df_national['Consumo_Tm'].mean():.0f} Tm")
print(f"   Consumption mínimo: {df_national['Consumo_Tm'].min():.0f} Tm")
print(f"   Consumption máximo: {df_national['Consumo_Tm'].max():.0f} Tm")
print(f"   Desv. estándar: {df_national['Consumo_Tm'].std():.0f} Tm")

print(f"\n📊 ESTACIONALIDAD:")
mes_max = df_national.groupby('Mes')['Consumo_Tm'].mean().idxmax()
mes_min = df_national.groupby('Mes')['Consumo_Tm'].mean().idxmin()
mes_nombres = ['January', 'February', 'March', 'April', 'May', 'June',
               'July', 'August', 'September', 'October', 'November', 'December']
print(f"   Mes con más consumption: {mes_nombres[mes_max-1]}")
print(f"   Mes con menos consumption: {mes_nombres[mes_min-1]}")

print(f"\n📅 TENDENCIA ANUAL:")
for año in sorted(df_national['Año'].unique()):
    consumption_año = df_national[df_national['Año'] == año]['Consumo_Tm'].sum()
    print(f"   {año}: {consumption_año:,.0f} Tm")

print(f"\n🔍 TRIMESTRES:")
for trim in sorted(df_national['Trimestre'].unique()):
    consumption_trim = df_national[df_national['Trimestre'] == trim]['Consumo_Tm'].mean()
    print(f"   Q{trim}: {consumption_trim:.0f} Tm (average)")

print(f"\n✅ EDA COMPLETED - Files saveds en /reports/figures/")
print("="*60)

## 6. Conclusions

- Explosive growth: ×80 en 3 años (2023-2025)
- Concentration in 5 CCAA: >90% of total consumption
- Clear seasonality: peak in Q4 (Oct-Dic)
- Madrid, Cataluña y Andalucía lead adoption